# CA-APSRG Retinal Vessel Segmentation - Google Colab Notebook

Notebook ini menjalankan project `ca_apsrg_retinal_project` di Google Colab.

Yang perlu disiapkan agar semua cell berjalan lengkap:

1. Repository GitHub berisi `../../app.py`, `../../requirements.txt`, `../../configs`, `../../src`, dan notebook ini.
2. Untuk demo single image: siapkan fundus image. Ground truth mask dan FoV mask bersifat optional.
3. Untuk viewer hasil eksperimen: commit folder `outputs/analysis*` dan `outputs/experiments*` yang ingin ditampilkan.
4. Runtime Colab cukup CPU. GPU tidak wajib untuk pipeline ini.

Catatan: dataset penuh tidak perlu masuk Colab. Untuk batch experiment besar, jalankan lokal lalu commit CSV/plot/output yang ingin dilihat di notebook atau Streamlit Cloud.

## 1. Setup Project

Mode default notebook ini adalah clone dari GitHub. Jika ingin upload ZIP project secara manual, ubah `USE_PROJECT_ZIP = True` dan `USE_GITHUB_REPO = False`.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import zipfile

USE_GITHUB_REPO = True
USE_PROJECT_ZIP = False
FORCE_RECLONE = False

REPO_URL = "https://github.com/nicolauseuclides512/ca_apsrg_retinal_project.git"
BRANCH = ""  # isi misalnya "main" kalau ingin branch tertentu
PROJECT_DIR = Path("/content/ca_apsrg_retinal_project")

if FORCE_RECLONE and PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

if USE_GITHUB_REPO and not PROJECT_DIR.exists():
    clone_cmd = ["git", "clone", "--depth", "1"]
    if BRANCH:
        clone_cmd.extend(["--branch", BRANCH])
    clone_cmd.extend([REPO_URL, str(PROJECT_DIR)])
    subprocess.run(clone_cmd, check=True)

if USE_PROJECT_ZIP and not PROJECT_DIR.exists():
    from google.colab import files
    print("Upload ZIP project, misalnya ca_apsrg_retinal_project.zip")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No ZIP uploaded")
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(zip_name, "r") as zf:
        zf.extractall("/content")
    if not PROJECT_DIR.exists():
        candidates = [p for p in Path("/content").iterdir() if p.is_dir() and (p / "src").exists()]
        if not candidates:
            raise FileNotFoundError("Tidak menemukan folder project yang berisi src/ setelah extract ZIP")
        PROJECT_DIR = candidates[0]

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f"Project folder not found: {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("Project root:", PROJECT_DIR)
print("Files:", [p.name for p in PROJECT_DIR.iterdir() if p.name in {"app.py", "requirements.txt", "configs", "src", "outputs"}])

## 2. Install Dependencies

Cell ini menginstall dependency dari `../../requirements.txt`. Jika Colab meminta restart runtime setelah install, restart runtime lalu jalankan ulang dari cell import.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Dependencies installed")

## 3. Imports dan Helper Colab

In [ ]:
import io
import json
from dataclasses import asdict, is_dataclass
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from PIL import Image, ImageOps

from src.evaluation.metrics import evaluate_segmentation
from src.preprocessing.preprocess import PreprocessConfig, preprocess_fundus
from src.segmentation.adaptive_morphology import AdaptiveMorphologyConfig
from src.segmentation.apsrg_baseline import APSRGParams, apsrg_segment
from src.segmentation.ca_apsrg import CAAPSRGConfig, ca_apsrg_refine
from src.segmentation.context_features import ContextFeatureConfig
from src.utils.image_io import overlay_mask_on_image, resize_if_needed

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

CONFIG_PATH = PROJECT_DIR / "configs" / "default.yaml"

def load_yaml_config(path: str | Path) -> dict[str, Any]:
    path = Path(path)
    if not path.is_file():
        return {}
    with path.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f) or {}

def ensure_display_mask(mask: np.ndarray | None) -> np.ndarray | None:
    if mask is None:
        return None
    arr = np.asarray(mask)
    if arr.ndim == 3:
        arr = arr[..., 0]
    if arr.dtype == np.bool_:
        return arr.astype(np.uint8) * 255
    if arr.size and float(np.nanmax(arr)) <= 1.0:
        return (arr > 0).astype(np.uint8) * 255
    return (arr > 127).astype(np.uint8) * 255

def pil_bytes_to_array(data: bytes, mode: str = "RGB") -> np.ndarray:
    with Image.open(io.BytesIO(data)) as img:
        img = ImageOps.exif_transpose(img)
        img = img.convert(mode)
        return np.asarray(img, dtype=np.uint8)

def upload_one_file(label: str, required: bool = False) -> tuple[str, bytes] | tuple[None, None]:
    if not IN_COLAB:
        raise RuntimeError("files.upload hanya tersedia di Google Colab")
    print(label)
    uploaded = files.upload()
    if not uploaded:
        if required:
            raise RuntimeError(f"File wajib belum diupload: {label}")
        return None, None
    name = next(iter(uploaded))
    return name, uploaded[name]

def display_grid(items: list[tuple[str, np.ndarray | str | Path | None]], cols: int = 3, figsize_per_item: float = 4.0) -> None:
    items = [(title, image) for title, image in items if image is not None]
    if not items:
        print("No images to display")
        return
    rows = int(np.ceil(len(items) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * figsize_per_item, rows * figsize_per_item))
    axes = np.asarray(axes).reshape(-1)
    for ax, (title, image) in zip(axes, items):
        if isinstance(image, (str, Path)):
            image = np.asarray(Image.open(image))
        ax.imshow(image, cmap="gray" if np.asarray(image).ndim == 2 else None)
        ax.set_title(title)
        ax.axis("off")
    for ax in axes[len(items):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

def to_plain_dict(value: Any) -> dict[str, Any]:
    if value is None:
        return {}
    if is_dataclass(value):
        return asdict(value)
    if isinstance(value, dict):
        return dict(value)
    if hasattr(value, "to_dict"):
        return dict(value.to_dict())
    return {}

def metrics_dataframe(baseline_metrics: Any, ca_metrics: Any) -> pd.DataFrame:
    baseline = to_plain_dict(baseline_metrics)
    ca = to_plain_dict(ca_metrics)
    rows = []
    for metric in ["accuracy", "precision", "recall", "specificity", "f1_score", "iou"]:
        b = baseline.get(metric)
        c = ca.get(metric)
        rows.append({"metric": metric, "APSRG": b, "CA-APSRG": c, "delta": None if b is None or c is None else c - b})
    return pd.DataFrame(rows)

print("Imports ready")

## 4. Single Image Demo

Upload fundus image. Ground truth dan FoV boleh dilewati dengan menekan cancel pada upload dialog.

In [ ]:
fundus_name, fundus_data = upload_one_file("Upload fundus image (png/jpg/jpeg/tif/tiff/ppm)", required=True)
gt_name, gt_data = upload_one_file("Optional: upload ground truth mask, atau cancel", required=False)
fov_name, fov_data = upload_one_file("Optional: upload FoV mask, atau cancel", required=False)

image_rgb = pil_bytes_to_array(fundus_data, mode="RGB")
gt_mask = ensure_display_mask(pil_bytes_to_array(gt_data, mode="L")) if gt_data is not None else None
fov_mask = ensure_display_mask(pil_bytes_to_array(fov_data, mode="L")) if fov_data is not None else None

print("Fundus:", fundus_name, image_rgb.shape)
print("Ground truth:", gt_name, None if gt_mask is None else gt_mask.shape)
print("FoV:", fov_name, None if fov_mask is None else fov_mask.shape)

display_grid([
    ("Original fundus", image_rgb),
    ("Ground truth", gt_mask),
    ("FoV mask", fov_mask),
], cols=3)

In [ ]:
config = load_yaml_config(CONFIG_PATH)

preprocess_cfg = PreprocessConfig.from_dict(config.get("preprocessing", {}))
apsrg_cfg = APSRGParams.from_dict(config.get("apsrg_baseline", {}))
adaptive_cfg = AdaptiveMorphologyConfig.from_dict(
    config.get("adaptive_morphology", {}),
    ca_apsrg_config=config.get("ca_apsrg", {}),
)
ca_cfg = CAAPSRGConfig.from_dict(config.get("ca_apsrg", {}))
context_cfg = ContextFeatureConfig.from_dict(config.get("context_features", {}))

preprocessed = preprocess_fundus(image_rgb, fov_mask=fov_mask, config=preprocess_cfg)
fov_for_segmentation = resize_if_needed(fov_mask, preprocessed, interpolation="nearest") if fov_mask is not None else None

baseline_mask, apsrg_debug = apsrg_segment(preprocessed, fov_mask=fov_for_segmentation, params=apsrg_cfg)
ca_mask, ca_debug = ca_apsrg_refine(
    baseline_mask,
    fov_mask=fov_for_segmentation,
    params=adaptive_cfg,
    ca_config=ca_cfg,
    context_config=context_cfg,
)

baseline_metrics = None
ca_metrics = None
if gt_mask is not None:
    baseline_metrics = evaluate_segmentation(
        baseline_mask,
        gt_mask,
        fov_for_segmentation,
        resize_prediction_to_gt=True,
        resize_fov_to_gt=True,
    )
    ca_metrics = evaluate_segmentation(
        ca_mask,
        gt_mask,
        fov_for_segmentation,
        resize_prediction_to_gt=True,
        resize_fov_to_gt=True,
    )

print("Segmentation done")

In [ ]:
baseline_overlay = overlay_mask_on_image(image_rgb, baseline_mask, alpha=0.45, mask_color=(255, 0, 0))
ca_overlay = overlay_mask_on_image(image_rgb, ca_mask, alpha=0.45, mask_color=(0, 255, 0))

display_grid([
    ("Original", image_rgb),
    ("Preprocessed", preprocessed),
    ("APSRG baseline", ensure_display_mask(baseline_mask)),
    ("CA-APSRG refined", ensure_display_mask(ca_mask)),
    ("APSRG overlay", baseline_overlay),
    ("CA-APSRG overlay", ca_overlay),
], cols=3)

display_grid([
    ("Vesselness", apsrg_debug.get("vesselness")),
    ("Seed map", ensure_display_mask(apsrg_debug.get("seeds"))),
    ("Candidate map", ensure_display_mask(apsrg_debug.get("candidate"))),
], cols=3)

if baseline_metrics is not None and ca_metrics is not None:
    display(metrics_dataframe(baseline_metrics, ca_metrics))
else:
    print("Ground truth tidak tersedia, metrik tidak dihitung.")

context_features = ca_debug.get("context_features", {})
selected_params = ca_debug.get("refinement_debug", {}).get("selected_parameters", {})
display(pd.DataFrame(
    list(context_features.items()) + [(f"selected_{k}", v) for k, v in selected_params.items()],
    columns=["item", "value"],
))

## 5. Batch / Experiment Result Viewer

Bagian ini membaca folder `outputs/analysis*` dan `outputs/experiments*` jika tersedia di repo atau ZIP project.

In [ ]:
EXPERIMENT_LABELS = {
    "exp1_recall_oriented": "Experiment 1 - Recall-oriented CA-APSRG",
    "exp2_precision_oriented": "Experiment 2 - Precision-oriented CA-APSRG",
    "exp3_balanced_main": "Experiment 3 - Balanced CA-APSRG Main Result",
    "exp4_static_morphology": "Experiment 4 - Static Morphology Ablation",
    "exp5_no_skeleton_guard": "Experiment 5 - No Skeleton Guard Ablation",
    "exp6_no_small_component": "Experiment 6 - No Small Component Removal Ablation",
}

def safe_read_csv(path: Path) -> pd.DataFrame | None:
    if not path.is_file():
        return None
    try:
        return pd.read_csv(path)
    except Exception as exc:
        print(f"Failed to read {path}: {exc}")
        return None

def discover_result_sets(outputs_dir: Path = PROJECT_DIR / "outputs") -> list[dict[str, Any]]:
    result_sets = []
    if (outputs_dir / "analysis").exists() or (outputs_dir / "experiments").exists():
        result_sets.append({
            "key": "default",
            "label": "Default Streamlit Result - Experiment 3 Copy",
            "analysis_dir": outputs_dir / "analysis",
            "experiments_dir": outputs_dir / "experiments",
        })
    for key, label in EXPERIMENT_LABELS.items():
        analysis_dir = outputs_dir / f"analysis_{key}"
        experiments_dir = outputs_dir / f"experiments_{key}"
        if analysis_dir.exists() or experiments_dir.exists():
            result_sets.append({"key": key, "label": label, "analysis_dir": analysis_dir, "experiments_dir": experiments_dir})
    return result_sets

def load_tables(result_set: dict[str, Any]) -> dict[str, pd.DataFrame]:
    table_specs = {
        "metrics_per_image": ("experiments_dir", "metrics_per_image.csv"),
        "metrics_summary": ("experiments_dir", "metrics_summary.csv"),
        "article_table_mean_std": ("analysis_dir", "article_table_mean_std.csv"),
        "improvement_by_dataset": ("analysis_dir", "improvement_by_dataset.csv"),
        "improvement_per_image": ("analysis_dir", "improvement_per_image.csv"),
        "summary_by_dataset_method": ("analysis_dir", "summary_by_dataset_method.csv"),
    }
    tables = {}
    for name, (root_key, filename) in table_specs.items():
        df = safe_read_csv(Path(result_set[root_key]) / filename)
        if df is not None:
            tables[name] = df
    return tables

result_sets = discover_result_sets()
display(pd.DataFrame([{k: str(v) if k.endswith("dir") else v for k, v in item.items()} for item in result_sets]))

In [ ]:
# Pilih result set. Ganti menjadi exp1_recall_oriented, exp2_precision_oriented, dst.
RESULT_SET_KEY = "default"

result_set = next((item for item in result_sets if item["key"] == RESULT_SET_KEY), None)
if result_set is None:
    raise ValueError(f"Result set not found: {RESULT_SET_KEY}")

tables = load_tables(result_set)
print(result_set["label"])
print("Available tables:", list(tables))

if "article_table_mean_std" in tables:
    display(tables["article_table_mean_std"])
if "improvement_by_dataset" in tables:
    cols = [c for c in ["dataset", "n_images", "delta_precision_mean", "delta_recall_mean", "delta_f1_score_mean", "delta_iou_mean"] if c in tables["improvement_by_dataset"].columns]
    display(tables["improvement_by_dataset"][cols])
if "metrics_summary" in tables:
    display(tables["metrics_summary"][[c for c in ["dataset", "method", "n_images", "precision_mean", "recall_mean", "f1_score_mean", "iou_mean", "accuracy_mean"] if c in tables["metrics_summary"].columns]])

In [ ]:
def plot_metric_comparison(summary_df: pd.DataFrame, metrics: list[str] = ["precision", "recall", "f1_score", "iou"]) -> None:
    if summary_df.empty or "dataset" not in summary_df.columns or "method" not in summary_df.columns:
        print("metrics_summary.csv tidak tersedia")
        return
    for metric in metrics:
        col = f"{metric}_mean"
        if col not in summary_df.columns:
            continue
        pivot = summary_df.pivot_table(index="dataset", columns="method", values=col, aggfunc="mean")
        ax = pivot.plot(kind="bar", figsize=(8, 4), ylim=(0, 1), title=f"{metric} comparison")
        ax.set_ylabel(metric)
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.show()

plot_metric_comparison(tables.get("metrics_summary", pd.DataFrame()))

plot_dir = Path(result_set["analysis_dir"]) / "plots"
plot_paths = sorted(plot_dir.glob("*.png")) if plot_dir.is_dir() else []
display_grid([(path.stem.replace("_", " ").title(), path) for path in plot_paths], cols=2, figsize_per_item=5.5)

## 6. Image Browser untuk Output Eksperimen

Gunakan `DATASET`, `IMAGE_ID`, dan `SORT_BY` untuk memilih contoh hasil.

In [ ]:
def is_missing(value: Any) -> bool:
    if value is None:
        return True
    if isinstance(value, float) and np.isnan(value):
        return True
    text = str(value).strip()
    return text == "" or text.lower() in {"nan", "none", "null"}

def resolve_artifact_path(value: Any, result_set: dict[str, Any]) -> Path | None:
    if is_missing(value):
        return None
    raw = str(value).strip()
    normalized = raw.replace("\\", "/")
    candidates = [Path(raw), Path(normalized), PROJECT_DIR / raw, PROJECT_DIR / normalized]
    parts = [p for p in normalized.split("/") if p and not p.endswith(":")]
    for marker in [PROJECT_DIR.name, "outputs", "data"]:
        if marker in parts:
            candidates.append(PROJECT_DIR / Path(*parts[parts.index(marker) + 1:]))
    for exp_dir_name in [Path(result_set["experiments_dir"]).name, "experiments"]:
        if exp_dir_name in parts:
            candidates.append(Path(result_set["experiments_dir"]) / Path(*parts[parts.index(exp_dir_name) + 1:]))
    for p in candidates:
        if p.is_file():
            return p
    return None

per_image = tables.get("metrics_per_image", tables.get("improvement_per_image", pd.DataFrame())).copy()
if per_image.empty:
    raise FileNotFoundError("metrics_per_image.csv atau improvement_per_image.csv tidak tersedia")

DATASET = "DRIVE"  # DRIVE, STARE, CHASEDB1
IMAGE_ID = None     # isi string image_id tertentu, atau None untuk otomatis
SORT_BY = "delta_f1_score"  # delta_f1_score, ca_apsrg_f1_score, image_id
ASCENDING = False

filtered = per_image[per_image["dataset"].astype(str) == DATASET].copy()
if filtered.empty:
    raise ValueError(f"Dataset tidak ditemukan: {DATASET}. Available: {sorted(per_image['dataset'].astype(str).unique())}")
if SORT_BY in filtered.columns:
    filtered[SORT_BY] = pd.to_numeric(filtered[SORT_BY], errors="coerce")
    filtered = filtered.sort_values(SORT_BY, ascending=ASCENDING, na_position="last")
else:
    filtered = filtered.sort_values("image_id")

if IMAGE_ID is None:
    row = filtered.iloc[0]
else:
    row = filtered[filtered["image_id"].astype(str) == IMAGE_ID].iloc[0]

print("Selected:", row["dataset"], row["image_id"])

metric_rows = []
for metric in ["accuracy", "precision", "recall", "specificity", "f1_score", "iou"]:
    b = row.get(f"baseline_{metric}")
    c = row.get(f"ca_apsrg_{metric}")
    d = row.get(f"delta_{metric}")
    if is_missing(d) and not is_missing(b) and not is_missing(c):
        d = float(c) - float(b)
    metric_rows.append({"metric": metric, "APSRG": b, "CA-APSRG": c, "delta": d})
display(pd.DataFrame(metric_rows))

artifact_columns = [
    ("Original fundus", "image_path"),
    ("Ground truth", "mask_path"),
    ("Preprocessed", "preprocessed_path"),
    ("APSRG baseline mask", "baseline_mask_path"),
    ("CA-APSRG mask", "ca_apsrg_mask_path"),
    ("APSRG overlay", "baseline_overlay_path"),
    ("CA-APSRG overlay", "ca_apsrg_overlay_path"),
    ("Side-by-side comparison", "comparison_path"),
]
images = []
missing = []
for title, col in artifact_columns:
    path = resolve_artifact_path(row.get(col), result_set)
    if path is None:
        if col in row.index and not is_missing(row.get(col)):
            missing.append(title)
        continue
    images.append((title, path))

display_grid(images, cols=2, figsize_per_item=5.5)
if missing:
    print("Artifacts not found:", missing)

## 7. Perbandingan Experiment 1-6

In [ ]:
def weighted_mean(df: pd.DataFrame, column: str) -> float | None:
    if df.empty or column not in df.columns:
        return None
    values = pd.to_numeric(df[column], errors="coerce")
    if "n_images" not in df.columns:
        return float(values.mean())
    weights = pd.to_numeric(df["n_images"], errors="coerce").fillna(0)
    valid = values.notna() & (weights > 0)
    if not valid.any():
        return float(values.mean())
    return float((values[valid] * weights[valid]).sum() / weights[valid].sum())

comparison_rows = []
for rs in [item for item in result_sets if item["key"] != "default"]:
    imp = safe_read_csv(Path(rs["analysis_dir"]) / "improvement_by_dataset.csv")
    summ = safe_read_csv(Path(rs["experiments_dir"]) / "metrics_summary.csv")
    if imp is None and summ is None:
        continue
    row = {"experiment": rs["label"]}
    if imp is not None:
        row["n_images"] = int(pd.to_numeric(imp.get("n_images", []), errors="coerce").sum())
        for metric in ["precision", "recall", "f1_score", "iou", "accuracy"]:
            row[f"delta_{metric}"] = weighted_mean(imp, f"delta_{metric}_mean")
    if summ is not None and "method" in summ.columns:
        method_text = summ["method"].astype(str).str.lower()
        ca = summ[method_text.str.contains("ca", na=False)]
        baseline = summ[(method_text == "apsrg") | method_text.str.contains("baseline", na=False)]
        for metric in ["precision", "recall", "f1_score", "iou", "accuracy"]:
            row[f"ca_{metric}"] = weighted_mean(ca, f"{metric}_mean")
            row[f"baseline_{metric}"] = weighted_mean(baseline, f"{metric}_mean")
    comparison_rows.append(row)

comparison = pd.DataFrame(comparison_rows)
display(comparison)

for metric in ["delta_precision", "delta_recall", "delta_f1_score", "delta_iou"]:
    if metric not in comparison.columns:
        continue
    ax = comparison.set_index("experiment")[[metric]].plot(kind="bar", figsize=(10, 4), legend=False, title=metric.replace("_", " ").title())
    ax.axhline(0, linewidth=1, color="black")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()